# File and Data Field Descriptions

- **train.csv** - Personal records for about two-thirds (~8700) of the passengers, to be used as training data.
  - **PassengerId** - A unique Id for each passenger. Each Id takes the form gggg\*pp where \*\*\_gggg indicates a group the passenger is travelling**\* with and **_pp is their number within the group_\*\*. People in a group are often family members, but not always.
  - **HomePlanet** - The planet the passenger departed from, typically their planet of permanent residence.
  - **CryoSleep** - Indicates whether the passenger elected to be put into suspended animation for the duration of the voyage. Passengers in cryosleep are confined to their cabins.
  - **Cabin** - The cabin number where the passenger is staying. Takes the form **_deck/num/side, where side can be either P for Port or S for Starboard_**.
  - **Destination** - The planet the passenger will be debarking to.
  - **Age** - The age of the passenger.
  - **VIP** - Whether the passenger has **_paid for special VIP service_** during the voyage.
  - **RoomService, FoodCourt, ShoppingMall, Spa, VRDeck** - Amount the passenger has billed at each of the Spaceship Titanic's many luxury amenities.
  - **Name** - The first and last names of the passenger.
  - **Transported** - Whether the passenger was transported to another dimension. **This is the target**, the column you are trying to predict.
- **test.csv** - Personal records for the remaining one-third (~4300) of the passengers, to be used as test data. Your task is to predict the value of Transported for the passengers in this set.
- **sample_submission.csv** - A submission file in the correct format.
  - **PassengerId** - Id for each passenger in the test set.
  - **Transported** - The target. For each passenger, predict either True or False.


**Todo:**

- Unbekannte Werte auffuellen
  - Name: werte koennen mit der PassengerId ergaenzt werden
  - HomePlanet: kann man dem haeufigsten vorkommenden ergaenzt werden "Eearth"
  - CryoSleep: Wenn Nachname gleich oder Cabin gleich dann Mehrheitsbestimmung, sonst "False" setzten.
- Kombiniere "RoomService, FoodCourt, ShoppingMall, Spa, VRDeck" zu Spalte "Total"
- one-hot:
  - VIP, da nur T/F
  - CryoSleep
  - HomePlanet -> droppen, da noise
  - Destination -> droppen, da noise
- Age zu bins
- Cabins:
  - Bei Duplikaten zu Spalte "FamilySize" hinzufügen
  - Deck koennte man binen
  - side zu one-hot
  - num koennte man droppen oder zu bins umwandeln
- Name: eventuell zu "isAlone" umwandeln und droppen
- PassengerId: unterteilen in Gruppe wie "\_gggg"


In [1]:
# data analysis and wrangling
import pandas as pd
import numpy as np
import random as rnd

# visualization
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

# machine learning
from sklearn.svm import SVC, LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import Perceptron, SGDClassifier, LogisticRegression
from xgboost import XGBClassifier
from sklearn.tree import DecisionTreeClassifier

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix


SEED = 42
np.random.seed(SEED)

In [ ]:
df_train = pd.read_csv("./data/spaceship-titanic/train.csv")
df_test = pd.read_csv("./data/spaceship-titanic/test.csv")

labels = df_train["Transported"]
# y_test = df_test["Transported"]

# df_train.drop(["Transported"], axis=1, inplace=True)
# df_test.drop(["Transported"], axis=1, inplace=True)

In [ ]:
labels

In [ ]:
combined = pd.concat([df_train, df_test], sort=False).reset_index(drop=True)
combined.isnull().sum()

In [ ]:
# Jeder Passanger ist einzigartig.
#
combined.select_dtypes(include=["O"]).nunique()

In [ ]:
combined.describe()

In [ ]:
labels.describe()

### Looking for ways to fill missing values


In [ ]:
pd.concat(g for _, g in combined.groupby("Name") if len(g) > 1)

In [ ]:
cabins = combined.copy()
# cabins["Deck"] = cabins["Cabin"].apply(lambda x: str(x).split("/")[0] if pd.notnull(x) else x)

cabins["FamilySize"] = cabins.groupby("Cabin")["Cabin"].transform("count")

In [ ]:
cabins[cabins["FamilySize"] > 1].sort_values("Cabin", ascending=False)

In [ ]:
combined[["CryoSleep", "Transported"]].groupby(["CryoSleep"], as_index=False).mean().sort_values(
    by="Transported", ascending=False
)

In [ ]:
combined[["CryoSleep", "Transported"]].groupby(["CryoSleep"], as_index=False).mean().sort_values(
    by="Transported", ascending=False
)

### Matplotlib experiments


In [ ]:
time = np.arange(0.0, 5.0, 0.2)  # (Beginn, End, Steps)
print(time)

In [ ]:
plt.plot(time, time**3, "r--")

In [ ]:
mu, sigma = 115, 15
x = mu + sigma * np.random.randn(10000)
fig, ax = plt.subplots(figsize=(5, 4), layout="constrained")
ax.hist(x, 50, density=True, facecolor="C0", alpha=0.75)

ax.set_xlabel("Length [cm]")
ax.set_ylabel("Probability")
ax.set_title("Aardvark lengths\n (not really)")
ax.text(75, 0.025, r"$\mu=115,\ \sigma=15$")
ax.axis([55, 175, 0, 0.03])
ax.grid(True)

In [ ]:
combined.head()

In [ ]:
for dataset in combined:
    print(dataset)

In [ ]:
df_train.info()

In [ ]:
df_train["Transported"] = df_train["Transported"].astype(int)

In [ ]:
# for dataset in combined:
#   dataset["Transported"] = dataset["Transported"].map({"True": 1, "False": 0}).astype(int)

In [ ]:
cats = df_train.select_dtypes(exclude=[np.number, "bool"])

In [ ]:
cats

In [ ]:
cats.drop(["Name"], axis=1, inplace=True)
cats.drop(["PassengerId"], axis=1, inplace=True)
cats.drop(["Cabin"], axis=1, inplace=True)

In [ ]:
num_comb = df_train.select_dtypes(include=[np.number])

In [ ]:
sns.heatmap(num_comb.corr(), annot=True, fmt=".2f")

In [ ]:
num_comb.corr()

In [ ]:
from scipy.stats import chi2_contingency


def cramers_v(x, y):
    confusion_matrix = pd.crosstab(x, y)
    chi2 = chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.sum().sum()
    r, k = confusion_matrix.shape
    return np.sqrt(chi2 / (n * (min(k - 1, r - 1))))


# cats = cats.columns.tolist()

corr_matrix = pd.DataFrame(index=cats, columns=cats)

for col1 in cats:
    for col2 in cats:
        corr_matrix.loc[col1, col2] = cramers_v(df_train[col1], df_train[col2])

corr_matrix = corr_matrix.astype(float)

# Heatmap visualisieren
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", vmin=0, vmax=1)
plt.title("Cramer V Korrelationsmatrix für kategoriale Variablen")
plt.show()